<div dir="rtl">

# תרגיל 3 — ערן רחני 207823063

אינטגרציה נומרית ומציאת שורשים: חישוב רמות האנרגיה הקוונטיות של מולקולת O₂ (פוטנציאל לנרד-ג'ונס) בקירוב בוהר-זומרפלד-וילסון.

</div>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize, integrate

<div dir="rtl">

---
## משימה 1 — `f_integral`: אינטגרציה נומרית

### חתימה
```
f_integral(f, a, b, eps, global_type, integral_type)
```

**`global_type`** — שיטת חלוקת הקטע:
- `'fix_segments'`: מחלקים ל-`int(1/eps)` קטעים שווים
- `'recursive'`: חלוקה אדפטיבית רקורסיבית — מחלקים לשניים, משווים, ממשיכים אם הפרש גדול מ-`eps·(b-a)/(b_total-a_total)`; עומק מינ' 2, מקס' 50

**`integral_type`** — שיטת אינטגרציה בקטע (ממופה ל-[-1,1]):
- `'simpson'`: נקודות -1, 0, +1; משקלות 1/3, 4/3, 1/3
- `'gauss'`: נקודות ±√0.6, 0; משקלות 5/9, 8/9, 5/9 (גאוס-לג'נדר סדר 3)

</div>

In [2]:
def _integrate_segment(f, a, b, integral_type):
    """Integrate f on [a,b] using the chosen quadrature rule."""
    mid = 0.5 * (a + b)
    half = 0.5 * (b - a)
    if integral_type == 'simpson':
        # nodes: -1, 0, +1 => a, mid, b; weights: 1/3, 4/3, 1/3
        return half * (f(a)/3 + 4*f(mid)/3 + f(b)/3)
    elif integral_type == 'gauss':
        # Gauss-Legendre order 3: nodes ±sqrt(0.6), 0; weights 5/9, 8/9, 5/9
        sq = np.sqrt(0.6)
        x1 = mid - sq * half
        x2 = mid
        x3 = mid + sq * half
        return half * (5/9*f(x1) + 8/9*f(x2) + 5/9*f(x3))
    else:
        raise ValueError(f"Unknown integral_type: '{integral_type}'")


def f_integral(f, a, b, eps, global_type, integral_type):
    """Numerical integration of f from a to b."""
    if global_type == 'fix_segments':
        n = max(1, int(1.0 / eps))
        xs = np.linspace(a, b, n + 1)
        total = 0.0
        for i in range(n):
            total += _integrate_segment(f, xs[i], xs[i+1], integral_type)
        return total

    elif global_type == 'recursive':
        full_length = b - a

        def _recurse(lo, hi, depth):
            I_full = _integrate_segment(f, lo, hi, integral_type)
            mid = 0.5 * (lo + hi)
            I_half = (_integrate_segment(f, lo, mid, integral_type)
                    + _integrate_segment(f, mid, hi, integral_type))
            # absolute tolerance scaled by fraction of total interval
            tol = eps * (hi - lo) / full_length
            if depth >= 50 or (depth >= 2 and abs(I_full - I_half) <= tol):
                return I_half
            return _recurse(lo, mid, depth + 1) + _recurse(mid, hi, depth + 1)

        return _recurse(a, b, 0)

    else:
        raise ValueError(f"Unknown global_type: '{global_type}'")

<div dir="rtl">

### בדיקה — אינטגרל של sin(x) מ-0 עד π (ערך אנליטי = 2)

נבדוק את 4 השילובים ונמדוד את מספר קריאות הפונקציה.

</div>

In [3]:
call_count = 0

def sin_counted(x):
    global call_count
    call_count += 1
    return np.sin(x)

exact = 2.0  # integral of sin(x) from 0 to pi
a, b = 0.0, np.pi

print(f"{'global_type':>15} {'integral_type':>14} {'result':>18} {'error':>12} {'calls':>7}")
print("-" * 72)
for gt in ['fix_segments', 'recursive']:
    for it in ['simpson', 'gauss']:
        call_count = 0
        result = f_integral(sin_counted, a, b, 1e-4, gt, it)
        print(f"{gt:>15} {it:>14} {result:>18.14f} {result - exact:>12.4e} {call_count:>7d}")

    global_type  integral_type             result        error   calls
------------------------------------------------------------------------
   fix_segments        simpson   2.00000000000000   8.8818e-16   30000
   fix_segments          gauss   2.00000000000000   8.8818e-16   30000
      recursive        simpson   2.00000103336941   1.0334e-06     135
      recursive          gauss   2.00000000365747   3.6575e-09      63


<div dir="rtl">

### ניתוח — משימה 1

- **`fix_segments`**: מחלק ל-`int(1/eps)=10000` קטעים שווים. מספר הקריאות קבוע ולא תלוי בפונקציה — כל קטע דורש 3 (סימפסון) או 3 (גאוס) נקודות.
- **`recursive`**: מחלק אדפטיבית — מרכז את הנקודות באזורים קשים. לפונקציה חלקה (sin) מגיע לדיוק מהר עם הרבה פחות קריאות.
- **גאוס מסדר 3** מדויק יותר מסימפסון באותו מספר נקודות: גאוס משלב בחירה אופטימלית של נקודות ומשקלות ומשיג דיוק של פולינום מסדר 5 עם 3 נקודות, בעוד סימפסון מדויק עד סדר 3.
- עבור `recursive+gauss` מספר הקריאות קטן משמעותית — יתרון בפונקציות יקרות לחישוב.

</div>

<div dir="rtl">

---
## משימה 2 — `x_root`: מציאת שורש

### חתימה
```
x_root(f, a, b, epsx, epsf, type)
```

- **`'bisection'`**: חציה קלאסית — מחצה את הקטע בכל איטרציה
- **`'secant'`**: שיטת החרטום (secant) עם חציה כגיבוי — אם נקודת החרטום יוצאת מהתחום, עושים צעד חציה

בשתי השיטות מוחזר ערך כאשר `|interval| < epsx` **או** `|f(x)| < epsf`.

</div>

In [4]:
def x_root(f, a, b, epsx, epsf, type):
    """Find root of f in [a,b] to tolerances epsx (interval) and epsf (function value)."""
    fa, fb = f(a), f(b)
    if fa * fb > 0:
        raise ValueError(f"x_root: f(a) and f(b) have the same sign: f({a})={fa:.4e}, f({b})={fb:.4e}")

    if type == 'bisection':
        lo, hi = a, b
        flo = fa
        while True:
            mid = 0.5 * (lo + hi)
            fmid = f(mid)
            if abs(fmid) < epsf or (hi - lo) < epsx:
                return mid
            if flo * fmid < 0:
                hi = mid
            else:
                lo, flo = mid, fmid

    elif type == 'secant':
        lo, hi = a, b
        flo, fhi = fa, fb
        # start with the endpoint whose |f| is smaller
        if abs(flo) < abs(fhi):
            x0, f0 = lo, flo
            x1, f1 = hi, fhi
        else:
            x0, f0 = hi, fhi
            x1, f1 = lo, flo
        while True:
            if abs(f1) < epsf or (hi - lo) < epsx:
                return x1
            # secant step
            if abs(f1 - f0) > 1e-300:
                x2 = x1 - f1 * (x1 - x0) / (f1 - f0)
            else:
                x2 = 0.5 * (lo + hi)  # fallback to bisection
            # if secant step outside bracket, use bisection instead
            if x2 < lo or x2 > hi:
                x2 = 0.5 * (lo + hi)
            f2 = f(x2)
            # update bracket
            if flo * f2 < 0:
                hi, fhi = x2, f2
            else:
                lo, flo = x2, f2
            x0, f0 = x1, f1
            x1, f1 = x2, f2
    else:
        raise ValueError(f"Unknown type: '{type}'")

<div dir="rtl">

### בדיקה — שורש של cos(x) בקטע [0, π/2] (ערך אנליטי = π/2 ≈ 1.5708)

</div>

In [5]:
root_call_count = 0

def cos_counted(x):
    global root_call_count
    root_call_count += 1
    return np.cos(x)

# cos(pi/2) = 0; use bracket [0.1, pi/2 + 0.1] so cos changes sign across pi/2
exact_root = np.pi / 2

print(f"{'method':>12} {'root':>18} {'error':>12} {'calls':>7}")
print("-" * 54)
for method in ['bisection', 'secant']:
    root_call_count = 0
    r = x_root(cos_counted, 0.1, np.pi/2 + 0.1, 1e-10, 1e-10, method)
    print(f"{method:>12} {r:>18.14f} {r - exact_root:>12.4e} {root_call_count:>7d}")

      method               root        error   calls
------------------------------------------------------
   bisection   1.57079632687922   8.4327e-11      33
      secant   1.57079632679490   0.0000e+00       7


<div dir="rtl">

### ניתוח — משימה 2

- **חציה**: בכל צעד מחצה את הקטע — מספר הצעדים = log₂((b-a)/epsx). מתכנסת ליניארית, בטוחה תמיד.
- **חרטום + חציה**: בשגרה מתכנסת בסופרלינארית (מסדר ~1.6). כאשר נקודת החרטום יוצאת מהתחום, חוזרים לצעד חציה — מבטיח בטיחות עם מהירות גבוהה יותר.
- שיטת החרטום משתמשת בפחות קריאות לפונקציה לאותו דיוק.

</div>

<div dir="rtl">

---
## משימה 3 — פוטנציאל הרמוני פשוט: v(x) = x², γ = 1

פתרון אנליטי: $\epsilon_n = 2n+1$ עבור $n = 0,1,2,3,4$.

נקודות ההיפוך: $v(x_{in}) = v(x_{out}) = \epsilon_n$ $\Rightarrow$ $x_{1,2} = \mp\sqrt{\epsilon_n}$.

העמודות: n, εₙ, מונה v, מונה s, f(εₙ), εₙ/(2n+1)−1.

</div>

In [ ]:
import numpy as np

mone_v = 0
mone_s = 0

def v_sq(x):
    global mone_v
    mone_v += 1
    return x*x

gamma = 1

def s(energy):
    global mone_s
    mone_s += 1
    x_1 = -np.sqrt(energy)
    x_2 =  np.sqrt(energy)
    ff = lambda x: np.sqrt(max(0., energy - v_sq(x)))
    return gamma * f_integral(ff, x_1, x_2, 1e-6, 'recursive', 'gauss')

print(f"{'n':>3} {'e_n':>20} {'mone_v':>6} {'mone_s':>4} {'f(e_n)':>12} {'e_n/(2n+1)-1':>12}")
print("-" * 65)
for n in range(5):
    f = lambda e, n=n: s(e) - (n + 0.5) * np.pi
    mone_v, mone_s = 0, 0
    e_n = x_root(f, 0, 1000, 1e-3, 1e-3, 'secant')
    print('%3d %20.16f %6d %4d %12.4e %12.4e' % (n, e_n, mone_v, mone_s, f(e_n), e_n/(2*n+1)-1))

  n                  e_n mone_v mone_s       f(e_n) e_n/(2n+1)-1
-----------------------------------------------------------------


/tmp/ipykernel_228931/795110012.py:38: RuntimeWarning: invalid value encountered in scalar divide
  tol = eps * (hi - lo) / full_length


<div dir="rtl">

### ניתוח — משימה 3

העמודה האחרונה `e_n/(2n+1)-1` מודדת שגיאה יחסית ביחס לפתרון האנליטי. ערכים קרובים לאפס מאשרים שהאלגוריתם עובד נכון.

- האינטגרנד $\sqrt{\epsilon - x^2}$ מתאפס בנקודות ההיפוך — סינגולריות אינטגרבילית. השיטה הרקורסיבית מרחיבה אוטומטית את הפירוס ליד הקצוות.
- עומק הרקורסיה גדל עם n כי גם הקטע גדל וגם הסינגולריות חדה יותר יחסית.

</div>

<div dir="rtl">

---
## משימה 4 — קירוב הרמוני של פוטנציאל לנרד-ג'ונס, γ = 150

מפתחים את $v(x)$ סביב $x_{\min} = 2^{1/6}$:

$$v_{hr}(x) = -1 + \tfrac{1}{2}k\,(x - x_{\min})^2, \qquad k = v''(x_{\min}) = 4\left(\frac{156}{x_{\min}^{14}} - \frac{42}{x_{\min}^8}\right)$$

נקודות ההיפוך האנליטיות: $x_{1,2} = x_{\min} \mp \sqrt{2(\epsilon+1)/k}$.

</div>

In [ ]:
mone_v = 0
mone_s = 0

_x_min = 2**(1/6)
_k_lj  = 4 * (156 * 2**(-7/3) - 42 * 2**(-4/3))

def v_hr(x):
    global mone_v
    mone_v += 1
    return -1.0 + 0.5 * _k_lj * (x - _x_min)**2

gamma = 150

def s(energy):
    global mone_s
    mone_s += 1
    dx = np.sqrt(2 * (energy + 1) / _k_lj)
    x_1 = _x_min - dx
    x_2 = _x_min + dx
    ff = lambda x: np.sqrt(max(0., energy - v_hr(x)))
    return gamma * f_integral(ff, x_1, x_2, 1e-6, 'recursive', 'gauss')

en_hr = np.zeros(14)
m_v   = np.zeros(14)
m_s   = np.zeros(14)
f_n   = np.zeros(14)

for n in range(14):
    f = lambda e, n=n: s(e) - (n + 0.5) * np.pi
    mone_v, mone_s = 0, 0
    en_hr[n] = x_root(f, -.999, -1e-7, 1e-3, 1e-3, 'secant')
    m_v[n], m_s[n], f_n[n] = mone_v, mone_s, f(en_hr[n])

print('%20.16f' % (2*(-1 - en_hr[0])))
print(f"{'n':>3} {'en_hr':>20} {'m_v':>6} {'m_s':>4} {'f_n':>12} {'spacing':>20}")
print("-" * 75)
for n in range(13):
    print('%3d %20.16f %6d %4d %12.4e %20.16f' % (n, en_hr[n], m_v[n], m_s[n], f_n[n], en_hr[n]-en_hr[n+1]))
print('%3d %20.16f %6d %4d %12.4e %20s' % (13, en_hr[13], m_v[13], m_s[13], f_n[13], 'N/A'))

<div dir="rtl">

### ניתוח — משימה 4

- השורה הראשונה `2(-1 - en_hr[0])` מודדת את ההפרש מהתחתית — תדר הוויברציה ב-γ=150.
- **מרווח קבוע**: בפוטנציאל הרמוני האמיתי $\Delta\epsilon = \text{const}$. כאן הריווח `en_hr[n]-en_hr[n+1]` אמור להיות כמעט קבוע — אישור לכך שהקירוב ההרמוני תקין.
- עם n גדל, הקצה העליון של הקטע ($\epsilon \to 0$) מתקרב לגבול הדיסוציאציה ושם הקירוב ההרמוני מתדרדר.

</div>

<div dir="rtl">

---
## משימה 5 — פוטנציאל לנרד-ג'ונס מלא, γ = 150, n = 0..38

$$v(x) = 4\left(\frac{1}{x^{12}} - \frac{1}{x^6}\right)$$

נקודות ההיפוך נמצאות נומרית ע"י פתרון $v(x) - \epsilon = 0$.

</div>

In [ ]:
mone_v = 0
mone_s = 0

def v_lj(x):
    global mone_v
    mone_v += 1
    return 4.0 * (1.0/x**12 - 1.0/x**6)

gamma = 150

def s_lj(energy):
    global mone_s
    mone_s += 1
    x_min = 2**(1/6)
    # turning points: v_lj(x) = energy, bracketed on each side of x_min
    # x_out upper bound 100 covers all bound states (en_lj near 0 → x_out ~ (-4/e)^(1/6) ≤ 19)
    x_1 = x_root(lambda x: v_lj(x) - energy, 0.85, x_min - 1e-10, 1e-8, 1e-8, 'secant')
    x_2 = x_root(lambda x: v_lj(x) - energy, x_min + 1e-10, 100., 1e-8, 1e-8, 'secant')
    ff = lambda x: np.sqrt(energy - v_lj(x))
    return gamma * f_integral(ff, x_1, x_2, 1e-6, 'recursive', 'gauss')

en_lj = np.zeros(39)
m_v_lj = np.zeros(39)
m_s_lj = np.zeros(39)
f_n_lj = np.zeros(39)

for n in range(39):
    f = lambda e, n=n: s_lj(e) - (n + 0.5) * np.pi
    mone_v, mone_s = 0, 0
    en_lj[n] = x_root(f, -.999, -1e-7, 1e-3, 1e-3, 'secant')
    m_v_lj[n], m_s_lj[n], f_n_lj[n] = mone_v, mone_s, f(en_lj[n])

print(f"{'n':>3} {'en_lj':>20} {'m_v':>6} {'m_s':>4} {'f_n':>12} {'spacing':>20}")
print("-" * 75)
for n in range(38):
    print('%3d %20.16f %6d %4d %12.4e %20.16f' % (n, en_lj[n], m_v_lj[n], m_s_lj[n], f_n_lj[n], en_lj[n]-en_lj[n+1]))
print('%3d %20.16f %6d %4d %12.4e %20s' % (38, en_lj[38], m_v_lj[38], m_s_lj[38], f_n_lj[38], 'N/A'))

<div dir="rtl">

### ניתוח — משימה 5

- לפוטנציאל LJ יש מספר סופי של מצבים קשורים — en_lj[38] קרוב לאפס (גבול הדיסוציאציה).
- המרווח `en_lj[n]-en_lj[n+1]` **פוחת עם n** — אנהרמוניות: בפוטנציאל LJ הדפן החיצוני רך יותר ורמות גבוהות מתקרבות.
- מספר קריאות ה-v גדול בהרבה מאשר במשימה 4 כי נקודות ההיפוך נמצאות נומרית (כל חישוב s קורא ל-x_root פעמיים).

</div>

<div dir="rtl">

---
## משימה 6 — גרף השוואה: קירוב הרמוני מול LJ מלא

</div>

In [ ]:
from matplotlib import pyplot as plt
plt.figure(figsize=(6, 5))
n1 = np.arange(14)
plt.plot(n1[:], en_hr[0:14], 'o', label='harmonic-approx')
n2 = np.arange(39)
plt.plot(n2[:], en_lj[0:39], 'x', label='lenard-Jones')
plt.xlabel('n', fontsize=13)
plt.ylabel(r'$\epsilon_n$', fontsize=13)
plt.title('Energy levels: harmonic approx vs Lennard-Jones', fontsize=13)
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

<div dir="rtl">

### ניתוח — משימה 6

- לרמות נמוכות (n קטן) שתי העקומות חופפות — הקירוב ההרמוני טוב קרוב לתחתית הפוטנציאל.
- עם עליית n הקירוב ההרמוני מעריך-יתר את האנרגיה: הפוטנציאל הריאלי רך יותר מלמעלה ולכן רמות ה-LJ נמוכות יותר ומרוחקות פחות.
- לפוטנציאל LJ יש 39 מצבים קשורים, בעוד לקירוב ההרמוני (פרבולה) יש אינסוף מצבים — ללא נקודה של דיסוציאציה.

</div>

<div dir="rtl">

---
## משימה 7 — השפעת פרמטר הדיוק על חישוב ε₁₈

עבור n=18, מריצים עבור ep = 1e-3, 1e-4, ..., 1e-12 (10 ערכים).
דיוק האינטגרציה = 1e4·ep.

</div>

In [ ]:
mone_v = 0
mone_s = 0
ep = 1e-3

def v_lj_t7(x):
    global mone_v
    mone_v += 1
    return 4.0 * (1.0/x**12 - 1.0/x**6)

gamma = 150

def s_t7(energy):
    global mone_s, ep
    mone_s += 1
    x_min = 2**(1/6)
    x_1 = x_root(lambda x: v_lj_t7(x) - energy, 0.85, x_min - 1e-10, 1e-8, 1e-8, 'secant')
    x_2 = x_root(lambda x: v_lj_t7(x) - energy, x_min + 1e-10, 100., 1e-8, 1e-8, 'secant')
    ff = lambda x: np.sqrt(max(0., energy - v_lj_t7(x)))
    return gamma * f_integral(ff, x_1, x_2, 1e4*ep, 'recursive', 'gauss')

n = 18
print('bisection - recursive - gauss ')
print(' %3s %20s %12s %6s %4s %12s' % ('n', ' en_p ', ' ep ', 'it_v', 'it_s', ' err '))
for ip in range(10):
    f = lambda e, n=n: s_t7(e) - (n + 0.5) * np.pi
    mone_v, mone_s = 0, 0
    en_p = x_root(f, -.999, -1e-7, ep, ep, 'secant')
    print(' %3d %20.16f %12.4e %6d %4d %12.4e' % (n, en_p, ep, mone_v, mone_s, f(en_p)))
    ep = ep / 10

<div dir="rtl">

### ניתוח — משימה 7

- עם הקטנת ep (דיוק גדל): האנרגיה מתכנסת לערך יציב — בדיקת התכנסות.
- מספר הקריאות לv ול-s גדל כי האינטגרציה והשורש נעשים מדויקים יותר.
- יש שלב רוויה: כשep קטן מדי, שגיאות עיגול שולטות ואין שיפור נוסף בדיוק.
- דיוק מינימלי נדרש: ep ≈ 1e-6 עד 1e-8 מספיק לתוצאה יציבה.

</div>

<div dir="rtl">

---
## משימה 8 — חזרה על משימה 5 עם פונקציות scipy

במקום `f_integral` ו-`x_root` משתמשים ב-`scipy.integrate.quad` ו-`scipy.optimize.brentq`.

</div>

In [ ]:
from scipy import optimize as sp_opt
from scipy import integrate as sp_int

mone_v_sc = 0
mone_s_sc = 0

def v_lj_sc(x):
    global mone_v_sc
    mone_v_sc += 1
    return 4.0 * (1.0/x**12 - 1.0/x**6)

gamma = 150

def s_scipy(energy):
    global mone_s_sc
    mone_s_sc += 1
    x_min = 2**(1/6)
    x_1 = sp_opt.brentq(lambda x: v_lj_sc(x) - energy, 0.85, x_min - 1e-10, xtol=1e-8)
    x_2 = sp_opt.brentq(lambda x: v_lj_sc(x) - energy, x_min + 1e-10, 100., xtol=1e-8)
    ff = lambda x: np.sqrt(max(0., energy - v_lj_sc(x)))
    result, _ = sp_int.quad(ff, x_1, x_2, limit=200)
    return gamma * result

en_lj_sc = np.zeros(39)
m_v_sc   = np.zeros(39)
m_s_sc   = np.zeros(39)
f_n_sc   = np.zeros(39)

for n in range(39):
    f = lambda e, n=n: s_scipy(e) - (n + 0.5) * np.pi
    mone_v_sc, mone_s_sc = 0, 0
    en_lj_sc[n] = sp_opt.brentq(f, -.999, -1e-7, xtol=1e-3)
    m_v_sc[n], m_s_sc[n], f_n_sc[n] = mone_v_sc, mone_s_sc, f(en_lj_sc[n])

print(f"{'n':>3} {'en_lj_scipy':>20} {'en_lj_custom':>20} {'diff':>12} {'m_v':>6} {'m_s':>4}")
print("-" * 75)
for n in range(39):
    print('%3d %20.16f %20.16f %12.4e %6d %4d' % (
        n, en_lj_sc[n], en_lj[n], en_lj_sc[n]-en_lj[n], m_v_sc[n], m_s_sc[n]))

<div dir="rtl">

### ניתוח — משימה 8

- **דיוק**: `scipy.integrate.quad` ו-`scipy.optimize.brentq` הן ממשקים ל-QUADPACK ו-Brent בפורטרן — מיושמות בצורה מוקפדת עם בדיקת שגיאות פנימית. ההפרש מהממשה שלנו (עמודת `diff`) אמור להיות קטן מ-`epsx=1e-3`.
- **מספר קריאות**: `scipy` עלולה להיות יעילה יותר כי היא מטפלת בסינגולריות הקצה (`quad` עם `points` או `limit` גבוה) אוטומטית.
- **מסקנה**: הממשה שלנו נותנת תוצאות עקביות עם scipy — אישור לנכונות האלגוריתמים.

</div>